# Optiver 介绍与两个方案

## 竞赛简介
Optiver 发起的 Kaggle 比赛 "Trading at the Close" 聚焦于预测美股收盘阶段的价格或方向性变化，以便在收盘竞价中做出更优交易决策。参赛者需要基于历史与当日的市场信息构建模型，输出在收盘窗口中的股票表现预测。

- 比赛主页: https://www.kaggle.com/competitions/optiver-trading-at-the-close
- 目标: 结合盘中与收盘前信息，对收盘时刻的价格变化或方向进行建模与预测
- 难点: 收盘阶段流动性与订单簿结构变化剧烈，时序与横截面信息需同时建模

## Notebook 内容结构
- 数据加载与基本检查
- 评估指标自动选择（回归/分类）
- 方案一：通用特征预处理 + 线性/逻辑回归基线
- 方案二：时序与分组特征工程 + 随机森林基线
- 交叉验证与方案对比
- 测试集推断与导出（如有）
- 结论与后续工作

## 运行环境
- Python 3.9+
- pandas, numpy, scikit-learn
- 可选：lightgbm 或 xgboost（本 Notebook 采用标准库可运行的基线方案）

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold, StratifiedKFold, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score, roc_auc_score, f1_score
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
np.random.seed(42)

## 数据加载

In [ ]:
DATA_DIRS = ["./data", "."]
TRAIN_FILES = ["train.csv", "optiver_train.csv"]
TEST_FILES = ["test.csv", "optiver_test.csv"]
def find_existing_path(candidates):
    for p in candidates:
        if os.path.exists(p):
            return p
    return None
def find_dataset_file(names):
    for d in DATA_DIRS:
        for n in names:
            p = os.path.join(d, n)
            if os.path.exists(p):
                return p
    return None
train_path = find_dataset_file(TRAIN_FILES)
test_path = find_dataset_file(TEST_FILES)
train_df = pd.read_csv(train_path) if train_path else None
test_df = pd.read_csv(test_path) if test_path else None
print("train_path", train_path)
print("test_path", test_path)
print(train_df.shape if train_df is not None else None)
print(test_df.shape if test_df is not None else None)

## 目标列与任务类型自动识别

In [ ]:
def detect_target(df):
    cols = df.columns.tolist()
    candidates = []
    for c in cols:
        cl = c.lower()
        if ("target" in cl) or ("movement" in cl) or (cl == "label") or (cl == "y"):
            candidates.append(c)
    if candidates:
        return candidates[0]
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    for c in numeric_cols[::-1]:
        if df[c].nunique() < len(df):
            return c
    return cols[-1]
def detect_task_type(y):
    uniq = pd.unique(y)
    if pd.api.types.is_numeric_dtype(y):
        ratio = len(uniq) / max(1, len(y))
        if len(uniq) <= 20 and ratio < 0.02:
            return "classification"
        return "regression"
    return "classification"
if train_df is not None:
    target_col = detect_target(train_df)
    y = train_df[target_col]
    task_type = detect_task_type(y)
    print("target_col", target_col)
    print("task_type", task_type)

## 特征与预处理

In [ ]:
def build_preprocessor(df, target):
    X = df.drop(columns=[target])
    num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = [c for c in X.columns.tolist() if c not in num_cols]
    transformers = []
    if num_cols:
        transformers.append(("num", StandardScaler(), num_cols))
    if cat_cols:
        transformers.append(("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols))
    return ColumnTransformer(transformers=transformers)
if train_df is not None:
    preprocessor = build_preprocessor(train_df, target_col)

## 方案一：通用基线（线性/逻辑回归）
思路：对数值特征做标准化、类别特征做独热编码，在此基础上使用 Ridge（回归）或 LogisticRegression（分类）进行交叉验证，作为强可解释的基线。

In [ ]:
def run_scheme1(df, target, task):
    X = df.drop(columns=[target])
    y = df[target]
    if task == "regression":
        model = Ridge(alpha=1.0)
        cv = KFold(n_splits=5, shuffle=True, random_state=42)
        pipe = Pipeline(steps=[("prep", build_preprocessor(df, target)), ("model", model)])
        scores = []
        for tr, va in cv.split(X):
            Xtr, Xva = X.iloc[tr], X.iloc[va]
            ytr, yva = y.iloc[tr], y.iloc[va]
            pipe.fit(Xtr, ytr)
            p = pipe.predict(Xva)
            mse = mean_squared_error(yva, p)
            r2 = r2_score(yva, p)
            scores.append((mse, r2))
        return scores, pipe
    else:
        model = LogisticRegression(max_iter=1000)
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        pipe = Pipeline(steps=[("prep", build_preprocessor(df, target)), ("model", model)])
        scores = []
        for tr, va in cv.split(X, y):
            Xtr, Xva = X.iloc[tr], X.iloc[va]
            ytr, yva = y.iloc[tr], y.iloc[va]
            pipe.fit(Xtr, ytr)
            proba = pipe.predict_proba(Xva)
            if proba.shape[1] == 2:
                auc = roc_auc_score(yva, proba[:, 1])
                scores.append(auc)
            else:
                pred = pipe.predict(Xva)
                f1 = f1_score(yva, pred, average="macro")
                scores.append(f1)
        return scores, pipe
scheme1_scores, scheme1_pipe = (None, None)
if train_df is not None:
    scheme1_scores, scheme1_pipe = run_scheme1(train_df, target_col, task_type)
    print("scheme1", scheme1_scores[:3])

## 方案二：时序与分组特征工程 + 随机森林
思路：利用典型交易数据结构（如 stock_id、date_id 等）进行分组统计与归一化特征构建，结合随机森林做非线性拟合，增强鲁棒性与泛化。

In [ ]:
def make_features(df, target):
    X = df.drop(columns=[target]).copy()
    num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    if "stock_id" in X.columns:
        g = X.groupby("stock_id")
        for c in num_cols:
            m = g[c].transform("mean")
            X[c + "_stock_mean"] = m
            X[c + "_rel"] = X[c] / (m + 1e-9)
    day_col = None
    for dc in ["date", "date_id", "day"]:
        if dc in X.columns:
            day_col = dc
            break
    if day_col is not None:
        g = X.groupby(day_col)
        for c in num_cols:
            m = g[c].transform("mean")
            X[c + "_day_mean"] = m
            X[c + "_z"] = (X[c] - m)
    return X
def run_scheme2(df, target, task):
    X = make_features(df, target)
    y = df[target]
    if task == "regression":
        model = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)
        cv = KFold(n_splits=5, shuffle=True, random_state=42)
        scores = []
        for tr, va in cv.split(X):
            Xtr, Xva = X.iloc[tr], X.iloc[va]
            ytr, yva = y.iloc[tr], y.iloc[va]
            model.fit(Xtr, ytr)
            p = model.predict(Xva)
            mse = mean_squared_error(yva, p)
            r2 = r2_score(yva, p)
            scores.append((mse, r2))
        return scores, model, X
    else:
        model = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        scores = []
        for tr, va in cv.split(X, y):
            Xtr, Xva = X.iloc[tr], X.iloc[va]
            ytr, yva = y.iloc[tr], y.iloc[va]
            model.fit(Xtr, ytr)
            proba = None
            try:
                proba = model.predict_proba(Xva)
            except Exception:
                proba = None
            if proba is not None and proba.shape[1] == 2:
                auc = roc_auc_score(yva, proba[:, 1])
                scores.append(auc)
            else:
                pred = model.predict(Xva)
                f1 = f1_score(yva, pred, average="macro")
                scores.append(f1)
        return scores, model, X
scheme2_scores, scheme2_model, scheme2_X = (None, None, None)
if train_df is not None:
    scheme2_scores, scheme2_model, scheme2_X = run_scheme2(train_df, target_col, task_type)
    print("scheme2", scheme2_scores[:3])

## 方案对比与选择

In [ ]:
def summarize_scores(task, scores):
    if scores is None:
        return None
    if task == "regression":
        mse_list = [s[0] for s in scores]
        r2_list = [s[1] for s in scores]
        return {"mse_mean": float(np.mean(mse_list)), "r2_mean": float(np.mean(r2_list))}
    else:
        return {"score_mean": float(np.mean(scores))}
s1 = summarize_scores(task_type, scheme1_scores)
s2 = summarize_scores(task_type, scheme2_scores)
print("scheme1_summary", s1)
print("scheme2_summary", s2)
best_scheme = None
if s1 is not None and s2 is not None:
    if task_type == "regression":
        best_scheme = "scheme1" if s1["mse_mean"] <= s2["mse_mean"] else "scheme2"
    else:
        best_scheme = "scheme1" if s1["score_mean"] >= s2["score_mean"] else "scheme2"
print("best_scheme", best_scheme)

## 测试集推断与提交文件（如有）

In [ ]:
def fit_full_and_predict(train_df, test_df, target, task, best):
    if best == "scheme1":
        pipe = Pipeline(steps=[("prep", build_preprocessor(train_df, target)), ("model", Ridge(alpha=1.0) if task == "regression" else LogisticRegression(max_iter=1000))])
        pipe.fit(train_df.drop(columns=[target]), train_df[target])
        if test_df is None:
            return None, None
        pred = pipe.predict(test_df)
        return pred, pipe
    elif best == "scheme2":
        X = make_features(train_df, target)
        if task == "regression":
            model = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)
        else:
            model = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
        model.fit(X, train_df[target])
        if test_df is None:
            return None, model
        Xt = make_features(pd.concat([train_df.drop(columns=[target]).iloc[:1]*0, test_df], axis=0, ignore_index=True).iloc[1:], target=None) if True else make_features(test_df.assign(dummy=0), target)
        pred = model.predict(Xt)
        return pred, model
    return None, None
pred, model = (None, None)
if train_df is not None and best_scheme is not None:
    pred, model = fit_full_and_predict(train_df, test_df, target_col, task_type, best_scheme)
    if pred is not None:
        id_col = None
        for c in ["row_id", "id"]:
            if test_df is not None and c in test_df.columns:
                id_col = c
                break
        sub = pd.DataFrame({id_col if id_col else "id": test_df[id_col] if id_col else np.arange(len(pred)), "prediction": pred})
        sub_path = "submission.csv"
        sub.to_csv(sub_path, index=False)
        print(sub_path)
    else:
        print(None)

## 结论与后续
- 方案一偏向稳健与可解释，适合快速建立基线与做错误分析
- 方案二通过分组与时序相关特征提升了表达能力，适合复杂市场结构
- 后续可引入更强模型（如 GBDT/Transformer）、更细粒度的时间窗特征与交易规则约束，进一步逼近收盘阶段的真实决策流程